# 🧬 **AI 3D MEDICAL X-RAY & BODY SCANNER PIPELINE (GOOGLE COLAB STUDIO)**
> **Cloud-Powered Video Production & High-Speed Rendering for YouTube Shorts & TikTok**

Notebook ini memungkinkan Anda memproduksi video 3D Medical berkelas **Zack D. Films** dengan kecepatan tinggi menggunakan server cloud Google (RAM 16GB+, multi-core CPU/GPU), tanpa membebani laptop dan baterai Anda.

---
### ⚡ **Fitur Utama:**
1. **Multi-Key Gemini Failover:** Mendukung hingga **4 Gemini API Key** sekaligus dengan rotasi otomatis (*Zero 429/503 errors*).
2. **Private & Public GitHub Support:** Mendukung repository Private via GitHub Token atau Public.
3. **AI Script & Timeline Guardian:** Menjamin hook 2.8 detik & total durasi Shorts 50–65 detik (90%+ retensi).
4. **Zack D. Films Macro Zooms & Dynamic Pathology:** Pembengkakan kandung kemih dinamis & refluks cairan urin ke ginjal.
5. **Turbo JPEG Rendering:** Render 2x lebih kencang menggunakan snapshot JPEG terkompresi & 100% core multi-thread.
6. **Auto-Save ke Google Drive & In-Browser Video Player.**

## ⚙️ **Langkah 1: Konfigurasi Parameter & Kunci API**
1. Masukkan **Topik Video** yang ingin dibuat.
2. Masukkan **Gemini API Key** (Mendukung hingga 4 key untuk rotasi otomatis).
3. Masukkan **Groq API Key** (Untuk sinkronisasi kata Whisper).
4. `GITHUB_TOKEN`: Biarkan kosong jika repo sudah Public.

In [ ]:
#@title 🎬 Pengaturan Topik & API Keys { display-mode: "form" }
TOPIK_VIDEO = "Apa yang Terjadi pada Tubuhmu Jika Menahan Kencing Terlalu Lama" #@param {type:"string"}
GITHUB_TOKEN = "" #@param {type:"string"}
GEMINI_API_KEY_1 = "" #@param {type:"string"}
GEMINI_API_KEY_2 = "" #@param {type:"string"}
GEMINI_API_KEY_3 = "" #@param {type:"string"}
GEMINI_API_KEY_4 = "" #@param {type:"string"}
GROQ_API_KEY = "" #@param {type:"string"}
SIMPAN_KE_GOOGLE_DRIVE = True #@param {type:"boolean"}

import os

try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get("GITHUB_TOKEN", "")
    GEMINI_API_KEY_1 = GEMINI_API_KEY_1 or userdata.get("GEMINI_API_KEY", "")
    GEMINI_API_KEY_2 = GEMINI_API_KEY_2 or userdata.get("GEMINI_API_KEY_2", "")
    GEMINI_API_KEY_3 = GEMINI_API_KEY_3 or userdata.get("GEMINI_API_KEY_3", "")
    GEMINI_API_KEY_4 = GEMINI_API_KEY_4 or userdata.get("GEMINI_API_KEY_4", "")
    GROQ_API_KEY = GROQ_API_KEY or userdata.get("GROQ_API_KEY", "")
except Exception:
    pass

os.environ["TOPIK_VIDEO"] = TOPIK_VIDEO
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
active_gemini_keys = [k for k in [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3, GEMINI_API_KEY_4] if k.strip()]

if GEMINI_API_KEY_1: os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY_1
if GEMINI_API_KEY_2: os.environ["GEMINI_API_KEY_2"] = GEMINI_API_KEY_2
if GEMINI_API_KEY_3: os.environ["GEMINI_API_KEY_3"] = GEMINI_API_KEY_3
if GEMINI_API_KEY_4: os.environ["GEMINI_API_KEY_4"] = GEMINI_API_KEY_4
if GROQ_API_KEY: os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("========================================================")
print(f"✓ Topik aktif         : \"{TOPIK_VIDEO}\"")
print(f"✓ Jumlah Gemini Key   : {len(active_gemini_keys)} Key terdeteksi (Multi-Key Failover Aktif)")
for idx, k in enumerate(active_gemini_keys, 1):
    print(f"  --> Key {idx} : {k[:6]}...{k[-4:]}")
print(f"✓ Status Groq API Key : {'Terpasang' if GROQ_API_KEY else 'Belum diisi (Gunakan .env repo)'}")
print(f"✓ Simpan Google Drive : {'Aktif' if SIMPAN_KE_GOOGLE_DRIVE else 'Nonaktif'}")
print("========================================================")

## 📦 **Langkah 2: Setup Environment & Download Repository**
Cell ini akan mengunduh repository, memasang Node.js 22, Puppeteer Chromium, FFmpeg, dan menyuntikkan API Keys.

In [ ]:
#@title 🚀 Install Dependencies & Setup Engine { display-mode: "form" }
import os
from google.colab import drive

# 1. Sambungkan Google Drive jika dipilih
if SIMPAN_KE_GOOGLE_DRIVE:
    print("📂 Menyambungkan Google Drive...")
    drive.mount("/content/drive")
    drive_out_dir = "/content/drive/MyDrive/Butterfly_Shorts_Output"
    os.makedirs(drive_out_dir, exist_ok=True)
    print(f"✓ Folder output Google Drive siap: {drive_out_dir}")

# 2. Clone atau Pull Repository Terbaru
%cd /content
repo_dir = "/content/butterfly-effect-shorts"
token = os.environ.get("GITHUB_TOKEN", "").strip()
repo_url = f"https://{token}@github.com/Kurniadinur/butterfly-effect-shorts.git" if token else "https://github.com/Kurniadinur/butterfly-effect-shorts.git"

if not os.path.exists(repo_dir):
    print("📥 Mengunduh repository dari GitHub...")
    !git clone {repo_url} butterfly-effect-shorts
    %cd /content/butterfly-effect-shorts
else:
    print("🔄 Memperbarui repository ke versi terbaru...")
    %cd /content/butterfly-effect-shorts
    !git pull origin main

# 3. Setup Node.js 22
print("\n⚙️ Memverifikasi versi Node.js...")
!node -v || (curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs)

# 4. Install Edge-TTS Python & System Codecs (Ubuntu Noble 24.04 Compatibility)
print("🎙️ Menginstall Edge-TTS & Codecs...")
!pip install -q edge-tts
!apt-get update -qq && (apt-get install -y -qq ffmpeg libgbm1 libnss3 libasound2t64 || apt-get install -y -qq ffmpeg libgbm1 libnss3 libasound2)

# 5. Install NPM Dependencies
print("📦 Menginstall NPM Packages...")
!npm install --legacy-peer-deps --no-audit --prefer-offline

# 6. Suntikkan Semua API Keys ke file .env
with open(".env", "a") as f:
    if GEMINI_API_KEY_1: f.write(f"\nGEMINI_API_KEY={GEMINI_API_KEY_1}\n")
    if GEMINI_API_KEY_2: f.write(f"\nGEMINI_API_KEY_2={GEMINI_API_KEY_2}\n")
    if GEMINI_API_KEY_3: f.write(f"\nGEMINI_API_KEY_3={GEMINI_API_KEY_3}\n")
    if GEMINI_API_KEY_4: f.write(f"\nGEMINI_API_KEY_4={GEMINI_API_KEY_4}\n")
    if GROQ_API_KEY: f.write(f"\nGROQ_API_KEY={GROQ_API_KEY}\n")

print("\n========================================================")
print("   ✅ SETUP ENGINE COLAB SELESAI & SIAP DIGUNAKAN!      ")
print("========================================================")

## 🧬 **Langkah 3: Jalankan AI Director Pipeline (Wajib Dijalankan agar Naskah Snappy 55s!)**
> ⚠️ **PENTING:** Selalu jalankan cell ini setiap kali ingin membuat video baru. **AI Timeline Guardian** akan memadatkan naskah menjadi **8 adegan berkecepatan tinggi (~1.600 frame / 55 detik)** sehingga render di Langkah 4 menjadi **3x lebih cepat**!

In [ ]:
#@title ⚡ Generate Naskah, Voiceover & Sinkronisasi Anatomi { display-mode: "form" }
%cd /content/butterfly-effect-shorts

topic_str = os.environ.get("TOPIK_VIDEO", "Apa yang Terjadi pada Tubuhmu Jika Menahan Kencing Terlalu Lama")
print(f"🎬 Memulai AI Director Pipeline untuk: \"{topic_str}\"...\n")

!node scripts/ai_director_pipeline.mjs "$TOPIK_VIDEO"


## 🏎️ **Langkah 4: Render Video MP4 Turbo (Turbo JPEG Engine)**
Merender video vertikal 1080x1920 @ 30fps dengan akselerasi `--image-format=jpeg --jpeg-quality=80` dan 100% core multi-threading.

In [ ]:
#@title 🎥 Render Video MP4 (Turbo High-Speed) { display-mode: "form" }
%cd /content/butterfly-effect-shorts
import os
import shutil
import time

os.makedirs("out", exist_ok=True)
output_mp4 = "out/butterfly_effect_shorts.mp4"

start_time = time.time()
print("🚀 Memulai proses render Remotion Turbo di Cloud...")

# Render Turbo: Concurrency 100% + JPEG snapshot (2x lebih kencang dibanding PNG default)
!npx remotion render src/index.ts ButterflyEffect-Shorts "$output_mp4" --concurrency=100% --image-format=jpeg --jpeg-quality=80 --gl=angle || \
 npx remotion render src/index.ts ButterflyEffect-Shorts "$output_mp4" --concurrency=100% --image-format=jpeg --jpeg-quality=80 --gl=swiftshader

elapsed = time.time() - start_time
if os.path.exists(output_mp4):
    size_mb = os.path.getsize(output_mp4) / (1024 * 1024)
    print(f"\n🎉 RENDER BERHASIL DALAM {elapsed:.1f} DETIK!")
    print(f"   Ukuran Video: {size_mb:.2f} MB")
    print(f"   Lokasi File : {output_mp4}")
    
    # Salin ke Google Drive jika aktif
    if SIMPAN_KE_GOOGLE_DRIVE and os.path.exists("/content/drive/MyDrive"):
        import re
        safe_topic = re.sub(r'[^a-zA-Z0-9_-]', '_', TOPIK_VIDEO)[:40]
        drive_dest = f"/content/drive/MyDrive/Butterfly_Shorts_Output/{safe_topic}.mp4"
        shutil.copy(output_mp4, drive_dest)
        print(f"\n💾 Video otomatis tersimpan aman di Google Drive Anda:")
        print(f"   --> {drive_dest}")
else:
    print("❌ Gagal merender video. Periksa log di atas.")

## 📺 **Langkah 5: Tonton Video Langsung di Colab & Download**
Anda bisa langsung menonton hasil video vertikalnya di bawah ini tanpa perlu mendownload terlebih dahulu.

In [ ]:
#@title 🎬 Preview Player & Download { display-mode: "form" }
from IPython.display import HTML
from base64 import b64encode
from google.colab import files
import os

output_mp4 = "/content/butterfly-effect-shorts/out/butterfly_effect_shorts.mp4"

if os.path.exists(output_mp4):
    mp4_data = open(output_mp4, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_data).decode()
    
    display(HTML(f"""
    <div style="display:flex; flex-direction:column; align-items:center; background:#0f172a; padding:24px; border-radius:16px;">
        <h3 style="color:#38bdf8; font-family:sans-serif; margin-bottom:12px;">📱 Preview Hasil Video YouTube Shorts (1080x1920)</h3>
        <video width="320" height="568" controls autoplay loop style="border-radius:12px; box-shadow:0 12px 36px rgba(0,0,0,0.8); border:2px solid #38bdf8;">
            <source src="{data_url}" type="video/mp4">
        </video>
    </div>
    """))
    
    # Opsi Download Langsung ke Laptop
    download_now = True #@param {type:"boolean"}
    if download_now:
        print("⬇️ Memulai pengunduhan file MP4 ke laptop Anda...")
        files.download(output_mp4)
else:
    print("⚠️ File video belum ditemukan. Silakan jalankan langkah render terlebih dahulu.")